# DeepfakeBench Score Extraction

This notebook builds DeepfakeBench-compatible dataset JSON from your frame manifest,
runs detector inference, and exports frame/video-level scores.


## 0. One-time setup on server

Run in terminal (outside notebook) if DeepfakeBench is not prepared yet:

```bash
git clone https://github.com/SCLBD/DeepfakeBench.git
cd DeepfakeBench
# Follow their install instructions (conda + install.sh) or your own env setup.
# Download pretrained detector weights and put them in ./training/weights
```


In [ ]:
from pathlib import Path
import json
import yaml
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader


In [ ]:
# =========================
# Configuration
# =========================
PROJECT_ROOT = Path("..").resolve()
DEEPFAKEBENCH_ROOT = Path("~/DeepfakeBench").expanduser().resolve()  # change if needed

# Input from your pipeline
SUBSET_NAME = "final"
FRAME_MANIFEST_CSV = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_frame_manifest.csv"

# DeepfakeBench detector config and weights
DETECTOR_CONFIG = DEEPFAKEBENCH_ROOT / "training/config/detector/xception.yaml"
WEIGHTS_PATH = DEEPFAKEBENCH_ROOT / "training/weights/xception_best.pth"  # change to actual weight file

# Custom dataset names/labels used inside DeepfakeBench JSON
CUSTOM_DATASET_NAME = "ThesisFinal"
CUSTOM_REAL_LABEL = "THESIS_real"
CUSTOM_FAKE_LABEL = "THESIS_fake"

# Output
OUT_DIR = PROJECT_ROOT / "outputs/deepfakebench_scores"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_JSON_DIR = OUT_DIR / "dataset_json"
DATASET_JSON_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_JSON_PATH = DATASET_JSON_DIR / f"{CUSTOM_DATASET_NAME}.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEEPFAKEBENCH_ROOT:", DEEPFAKEBENCH_ROOT)
print("FRAME_MANIFEST_CSV:", FRAME_MANIFEST_CSV)
print("DETECTOR_CONFIG:", DETECTOR_CONFIG)
print("WEIGHTS_PATH:", WEIGHTS_PATH)


In [ ]:
assert DEEPFAKEBENCH_ROOT.exists(), f"DeepfakeBench root not found: {DEEPFAKEBENCH_ROOT}"
assert FRAME_MANIFEST_CSV.exists(), f"Frame manifest not found: {FRAME_MANIFEST_CSV}"
assert DETECTOR_CONFIG.exists(), f"Detector config not found: {DETECTOR_CONFIG}"
assert WEIGHTS_PATH.exists(), f"Weights not found: {WEIGHTS_PATH}"


In [ ]:
# Build DeepfakeBench JSON for a custom dataset from your frame manifest
frame_df = pd.read_csv(FRAME_MANIFEST_CSV)

required_cols = {"video_id", "frame_path", "label", "split"}
missing = required_cols - set(frame_df.columns)
assert not missing, f"Missing required columns in frame manifest: {missing}"

work_df = frame_df.copy()
work_df["label_norm"] = work_df["label"].astype(str).str.lower()
work_df["split_norm"] = work_df["split"].astype(str).str.lower().replace({"valid": "val", "validation": "val"})
work_df = work_df[work_df["split_norm"].isin(["train", "val", "test"])].copy()

# map your labels to custom DeepfakeBench labels
work_df["dfb_label"] = np.where(work_df["label_norm"].eq("fake"), CUSTOM_FAKE_LABEL, CUSTOM_REAL_LABEL)

dataset_dict = {
    CUSTOM_DATASET_NAME: {
        CUSTOM_REAL_LABEL: {"train": {}, "val": {}, "test": {}},
        CUSTOM_FAKE_LABEL: {"train": {}, "val": {}, "test": {}},
    }
}

for (dfb_label, split, video_id), g in work_df.groupby(["dfb_label", "split_norm", "video_id"], sort=False):
    frames = sorted(g["frame_path"].astype(str).tolist())
    if not frames:
        continue
    dataset_dict[CUSTOM_DATASET_NAME][dfb_label][split][str(video_id)] = {
        "label": dfb_label,
        "frames": frames,
    }

with open(CUSTOM_JSON_PATH, "w") as f:
    json.dump(dataset_dict, f)

print("Saved custom dataset JSON:", CUSTOM_JSON_PATH)
print("Videos per label/split:")
for label_name in [CUSTOM_REAL_LABEL, CUSTOM_FAKE_LABEL]:
    for split in ["train", "val", "test"]:
        n = len(dataset_dict[CUSTOM_DATASET_NAME][label_name][split])
        print(f"  {label_name:12s} | {split:5s}: {n}")


In [ ]:
# Import DeepfakeBench modules
import sys
sys.path.insert(0, str(DEEPFAKEBENCH_ROOT / "training"))

from dataset.abstract_dataset import DeepfakeAbstractBaseDataset
from detectors import DETECTOR


In [ ]:
# Build runtime config (detector yaml + test config + overrides)
with open(DETECTOR_CONFIG, "r") as f:
    cfg = yaml.safe_load(f)
with open(DEEPFAKEBENCH_ROOT / "training/config/test_config.yaml", "r") as f:
    test_cfg = yaml.safe_load(f)
cfg.update(test_cfg)

cfg["test_dataset"] = [CUSTOM_DATASET_NAME]
cfg["dataset_json_folder"] = str(DATASET_JSON_DIR)
cfg["weights_path"] = str(WEIGHTS_PATH)
cfg["lmdb"] = False
cfg["workers"] = int(cfg.get("workers", 4))
cfg["cuda"] = bool(torch.cuda.is_available())
cfg["label_dict"] = {
    CUSTOM_REAL_LABEL: 0,
    CUSTOM_FAKE_LABEL: 1,
}

print("Model name:", cfg["model_name"])
print("CUDA available:", torch.cuda.is_available())
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Inference loop (equivalent to test.py, but we keep predictions for CSV export)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_cfg_single = cfg.copy()
test_cfg_single["test_dataset"] = CUSTOM_DATASET_NAME

test_set = DeepfakeAbstractBaseDataset(config=test_cfg_single, mode="test")
test_loader = DataLoader(
    dataset=test_set,
    batch_size=cfg["test_batchSize"],
    shuffle=False,
    num_workers=int(cfg["workers"]),
    collate_fn=test_set.collate_fn,
    drop_last=False,
)

model_class = DETECTOR[cfg["model_name"]]
model = model_class(cfg).to(device)
ckpt = torch.load(cfg["weights_path"], map_location=device)
model.load_state_dict(ckpt, strict=True)
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, total=len(test_loader), desc="DeepfakeBench inference"):
        labels = torch.where(batch["label"] != 0, 1, 0)
        batch["image"] = batch["image"].to(device)
        batch["label"] = labels.to(device)
        if batch["mask"] is not None:
            batch["mask"] = batch["mask"].to(device)
        if batch["landmark"] is not None:
            batch["landmark"] = batch["landmark"].to(device)

        pred = model(batch, inference=True)
        probs = pred["prob"].detach().cpu().numpy().reshape(-1)
        labs = batch["label"].detach().cpu().numpy().reshape(-1)

        all_probs.extend(probs.tolist())
        all_labels.extend(labs.tolist())

frame_paths = test_set.data_dict["image"]
assert len(frame_paths) == len(all_probs), "Mismatch between number of frames and scores"

frame_scores_df = pd.DataFrame({
    "frame_path": pd.Series(frame_paths, dtype=str),
    "detector_score": np.array(all_probs, dtype=float),
    "label": np.array(all_labels, dtype=int),
})

# video_id is parent folder name in your preprocessing output
frame_scores_df["video_id"] = frame_scores_df["frame_path"].apply(lambda p: Path(p).parent.name)

video_scores_df = (
    frame_scores_df
    .groupby("video_id", as_index=False)
    .agg(
        detector_score=("detector_score", "mean"),
        n_frames=("frame_path", "count"),
        label=("label", "max"),
    )
)

frame_out = OUT_DIR / f"{CUSTOM_DATASET_NAME}_{cfg['model_name']}_frame_scores.csv"
video_out = OUT_DIR / f"{CUSTOM_DATASET_NAME}_{cfg['model_name']}_video_scores.csv"
frame_scores_df.to_csv(frame_out, index=False)
video_scores_df.to_csv(video_out, index=False)

print("Saved frame scores:", frame_out)
print("Saved video scores:", video_out)
print(video_scores_df.head())


In [ ]:
# Optional: merge with your detector_processed format
# If needed by stage4b, keep only video_id + detector_score columns.

stage4b_out = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_detector_scores_deepfakebench.csv"
stage4b_out.parent.mkdir(parents=True, exist_ok=True)

video_scores_df[["video_id", "detector_score"]].to_csv(stage4b_out, index=False)
print("Saved stage4b-compatible CSV:", stage4b_out)
